## Colab setup

Run this cell **first**. It clones the repository, installs dependencies,
copies the raw workbook across from Drive, and points the notebooks at the
clone.

**Put your GitHub token in Colab's secrets panel** (the key icon in the left
sidebar), named `GH_TOKEN`, with "Notebook access" switched on. Do not paste
it into a cell — a pasted token gets pushed to GitHub and GitHub will revoke
it automatically.

Safe to re-run. Outside Colab (local Jupyter) it does nothing, so the same
notebook works in both places.

**Colab wipes `/content` when the runtime disconnects.** Push before you
close the tab, or the run is lost — see the last cell of this notebook.


In [ ]:
# ============================================================
# COLAB SETUP — run first. No-op outside Colab. Safe to re-run.
# ============================================================
import os, sys, subprocess
from pathlib import Path

GH_USER    = "KT-Devv"
GH_REPO    = "student-dropout-prediction-ghana"
GH_BRANCH  = "main"
DRIVE_XLSX = "/content/drive/MyDrive/Ghana_Dropout_Project/ghana_dropout_study_M.xlsx"

GIT_NAME   = "Your Name"          # <-- edit
GIT_EMAIL  = "your@email"         # <-- edit

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if not IN_COLAB:
    print("Not in Colab — skipping setup. Paths resolve from the repo root.")
else:
    def sh(cmd, check=True):
        r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
        if r.stdout.strip(): print(r.stdout.strip()[:2000])
        if check and r.returncode != 0:
            print(r.stderr.strip()[:2000])
        return r

    # ---- token from the secrets panel, never from a pasted string -------
    TOKEN = None
    try:
        from google.colab import userdata
        TOKEN = userdata.get("GH_TOKEN")
    except Exception:
        pass
    if not TOKEN:
        print("No GH_TOKEN secret found. Cloning read-only — you will be able "
              "to run, but NOT push.\n"
              "Add it: key icon in the left sidebar -> GH_TOKEN -> "
              "Notebook access on.")

    # ---- clone (or reuse an existing clone) -----------------------------
    REPO_PATH = Path(f"/content/{GH_REPO}")
    if REPO_PATH.exists():
        print(f"Repo already present at {REPO_PATH} — pulling latest.")
        sh(f"git -C {REPO_PATH} pull --ff-only", check=False)
    else:
        url = (f"https://{TOKEN}@github.com/{GH_USER}/{GH_REPO}.git" if TOKEN
               else f"https://github.com/{GH_USER}/{GH_REPO}.git")
        r = sh(f"git clone -b {GH_BRANCH} {url} {REPO_PATH}", check=False)
        if not REPO_PATH.exists():
            raise RuntimeError(
                "Clone failed. Check GH_USER/GH_REPO/GH_BRANCH above, and that "
                "your GH_TOKEN has Contents: Read and write on this repository."
            )

    os.chdir(REPO_PATH)
    os.environ["DROPOUT_REPO"] = str(REPO_PATH)
    if str(REPO_PATH) not in sys.path:
        sys.path.insert(0, str(REPO_PATH))

    # ---- dependencies ---------------------------------------------------
    if Path("requirements.txt").exists():
        print("installing requirements (quiet, ~1-2 min on a cold runtime)...")
        sh("pip install -q -r requirements.txt", check=False)

    # ---- raw data: the ONLY thing Drive is used for ---------------------
    # Pupil-level data is never committed (ethics: HuSSREC/AP/543/VOL. 5),
    # so it is copied in at runtime and .gitignore keeps it out of git.
    Path("data-raw").mkdir(exist_ok=True)
    target = Path("data-raw") / Path(DRIVE_XLSX).name
    if target.exists():
        print(f"raw workbook already present: {target}")
    else:
        try:
            from google.colab import drive
            if not os.path.exists("/content/drive/MyDrive"):
                drive.mount("/content/drive")
            if os.path.exists(DRIVE_XLSX):
                sh(f'cp "{DRIVE_XLSX}" data-raw/')
                print(f"copied raw workbook -> {target}")
            else:
                print(f"NOT FOUND: {DRIVE_XLSX}\n"
                      "Fix DRIVE_XLSX above, or upload the workbook to "
                      "data-raw/ manually. Notebooks 2-9 don't need it "
                      "(they read data-processed/cleaned_data.csv).")
        except Exception as e:
            print("Drive mount skipped:", e)

    # ---- git identity, needed before any commit -------------------------
    sh(f'git config user.name "{GIT_NAME}"', check=False)
    sh(f'git config user.email "{GIT_EMAIL}"', check=False)

    # ---- the check worth not skipping -----------------------------------
    r = subprocess.run("git status --porcelain", shell=True, text=True,
                       capture_output=True)
    leaked = [l for l in r.stdout.splitlines()
              if "data-raw" in l or "ghana_dropout_study" in l
              or "cleaned_data.csv" in l]
    if leaked:
        print("\n*** WARNING: pupil-level data is NOT being ignored by git ***")
        for l in leaked: print("   ", l)
        print("Do not commit until .gitignore covers these.")
    else:
        print("\ngit is correctly ignoring the raw data.")

    print(f"\nREPO : {os.getcwd()}")
    print(f"push : {'enabled' if TOKEN else 'DISABLED (no GH_TOKEN)'}")


# Notebook 4 — Baseline Classifiers

## What changed from R01

| Change | Reason |
|---|---|
| `cross_validate` no longer runs on the full `X, y` | R01 cell 15 cross-validated over all 1000 rows, so the test partition was inside the CV training folds. All CV is now on the training pool |
| **No test-set scoring at all here** | GATE-1(iii) — the same seed-42 partition was scored in Notebooks 4, 5b, 6, 6b, 7 and 8. The test set is now scored once, in Notebook 8 |
| Preprocessing via `preprocess_inside_fold` | Same pipeline as every other notebook, so the six-model comparison is commensurable with the focal-loss experiment |
| Primary metric is **AUC-PR**, ranking on AUC-PR | R01 ranked and selected on F1 while M15 declared AUC-PR primary |
| Per-fold scores committed | Q8 — needed for paired tests and variance |
| Equal budget stated explicitly, zero trials each | GATE-2 disclosure |

Output feeds **Supplementary Table S1**, which objective 2 promises and R01
never supplied. The data existed in `baseline_results.csv` all along, so
supplying it is the cheap route to closing half that objective.

In [ ]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      score_binary, two_level_variance, raw_feature_cols)

banner("NOTEBOOK 4 — BASELINES")
OUT = run_dir("notebook04_baselines")
capture_environment(OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)
print(f"train_pool {train_pool.shape}  |  test NOT touched in this notebook")

BASELINE_SEEDS = SEEDS[:3]     # 3 seeds x 25 folds x 6 models; widen if you want
print(f"seeds: {BASELINE_SEEDS}, folds per seed: {N_SPLITS*N_REPEATS}")

In [ ]:
# ---- model pool ---------------------------------------------------------
# All six receive: the identical in-fold pipeline, the identical folds, and
# ZERO hyperparameter search trials. That is parity at the floor, and it is
# stated rather than implied (GATE-2).
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def make_models(seed):
    m = {
        "Logistic Regression": LogisticRegression(
            random_state=seed, max_iter=2000, class_weight="balanced"),
        "Decision Tree": DecisionTreeClassifier(
            random_state=seed, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(
            random_state=seed, n_estimators=200, class_weight="balanced"),
    }
    try:
        from xgboost import XGBClassifier
        m["XGBoost"] = XGBClassifier(random_state=seed, eval_metric="logloss",
                                     verbosity=0)
    except ImportError:
        print("  xgboost unavailable — skipped (state this in M11)")
    try:
        from lightgbm import LGBMClassifier
        # SHARED_PARAMS already carries verbosity; passing it again is a
        # duplicate-keyword TypeError.
        m["LightGBM"] = LGBMClassifier(objective="binary", random_state=seed,
                                       **SHARED_PARAMS)
    except ImportError:
        print("  lightgbm unavailable — skipped")
    try:
        from catboost import CatBoostClassifier
        m["CatBoost"] = CatBoostClassifier(random_state=seed, verbose=0,
                                           allow_writing_files=False)
    except ImportError:
        print("  catboost unavailable — skipped (state this in M11)")
    return m

NEEDS_SCALING = {"Logistic Regression"}
print("models:", list(make_models(42)))

In [ ]:
# ---- run: in-fold preprocessing, training pool only --------------------
import time
rows = []
t0 = time.perf_counter()
for seed in BASELINE_SEEDS:
    for fi, (tr, vl) in enumerate(cv_splits(train_pool, seed), 1):
        X_tr, y_tr, X_vl, y_vl, meta = preprocess_inside_fold(
            train_pool.iloc[tr], train_pool.iloc[vl])
        for name, model in make_models(seed).items():
            A, B = X_tr, X_vl
            if name in NEEDS_SCALING:
                sc = StandardScaler().fit(X_tr)          # fit on train fold only
                A = pd.DataFrame(sc.transform(X_tr), columns=X_tr.columns)
                B = pd.DataFrame(sc.transform(X_vl), columns=X_vl.columns)
            t = time.perf_counter()
            model.fit(A, y_tr)
            fit_s = time.perf_counter() - t
            p = model.predict_proba(B)[:, 1]
            rows.append({"seed": seed, "fold": fi, "arm": name,
                         "search_trials": 0, "n_features": X_tr.shape[1],
                         "fit_seconds": fit_s, **score_binary(y_vl, p)})
    print(f"  seed {seed} done [{time.perf_counter()-t0:.0f}s]")

fold_df = pd.DataFrame(rows)
fold_df.to_csv(OUT / "baseline_fold_scores.csv", index=False)
print(f"\n{len(fold_df)} fold-level rows")

In [ ]:
# ---- Supplementary Table S1 (the table objective 2 promised) -----------
var = two_level_variance(fold_df, PRIMARY_METRIC)

agg = (fold_df.groupby("arm")
       .agg(auc_pr_mean=("auc_pr", "mean"),
            auc_roc_mean=("auc_roc", "mean"),
            recall_mean=("recall", "mean"),
            precision_mean=("precision", "mean"),
            macro_f1_mean=("macro_f1", "mean"),
            accuracy_mean=("accuracy", "mean"),
            mean_fit_seconds=("fit_seconds", "mean"))
       .reset_index()
       .merge(var[["arm", "between_seed_sd", "mean_within_seed_fold_sd"]], on="arm")
       .sort_values("auc_pr_mean", ascending=False))
agg["search_trials"] = 0
agg["n_seeds"] = len(BASELINE_SEEDS)
agg["n_folds_per_seed"] = N_SPLITS * N_REPEATS
agg["base_rate_pct"] = round(100 * float(train_pool[TARGET].mean()), 1)
agg["n_positive_train_pool"] = int(train_pool[TARGET].sum())
agg.to_csv(OUT / "supplementary_table_S1.csv", index=False)

print("SUPPLEMENTARY TABLE S1 — six-classifier comparison")
print("(cross-validated on the training pool; NO test-set figures)\n")
print(agg.drop(columns=["accuracy_mean"]).round(4).to_string(index=False))
print(f"\nRanked on {PRIMARY_METRIC}, matching M15. R01 ranked on F1.")
print(f"Base rate {agg['base_rate_pct'].iloc[0]}% on "
      f"{agg['n_positive_train_pool'].iloc[0]} positive cases — print both "
      "beside every figure in this table (J4).")

plt.figure(figsize=(9, 4.5))
o = agg.sort_values("auc_pr_mean")
plt.barh(o["arm"], o["auc_pr_mean"],
         xerr=o["between_seed_sd"], color="steelblue")
plt.axvline(float(train_pool[TARGET].mean()), ls="--", c="r", lw=1,
            label="base rate (uninformative)")
plt.xlabel("AUC-PR (mean over seeds, error bars = between-seed SD)")
plt.legend(); plt.tight_layout()
plt.savefig(OUT / "figures/baseline_comparison.png", dpi=200); plt.close()

In [ ]:
# ---- substrate justification (M11) -------------------------------------
print("VARIANCE, BOTH LEVELS (J3 — separately AND comparably):")
print(var.round(4).to_string(index=False))

best = agg.iloc[0]
print(f"\nstrongest baseline on {PRIMARY_METRIC}: {best['arm']} "
      f"({best['auc_pr_mean']:.4f})")
if "LightGBM" in set(agg["arm"]):
    lg = agg[agg["arm"] == "LightGBM"].iloc[0]
    print(f"LightGBM: {lg['auc_pr_mean']:.4f} "
          f"(gap to best: {best['auc_pr_mean']-lg['auc_pr_mean']:+.4f}, "
          f"between-seed SD {lg['between_seed_sd']:.4f})")
    print("\nM11's justification stands on two legs and should say both: "
          "LightGBM is competitive with the strongest baseline (quantified "
          "above), AND it is the only one of the six that accepts a fully "
          "custom training objective — which the intervention requires.")

write_manifest(OUT, {"notebook": "04_baselines",
                     "test_set_scored": False,
                     "seeds": BASELINE_SEEDS,
                     "models": list(agg["arm"]),
                     "search_trials_per_model": 0,
                     "ranking_metric": PRIMARY_METRIC})
print("\nNEXT: Notebook 5 (class imbalance, resampling inside the fold).")

---

## Save your work

Colab wipes `/content` when the runtime disconnects. Run this before you
close the tab — including `results/`, which has to be committed (the previous
review failed partly because the diagnostic CSVs backing the reported tables
were not in the repository).


In [ ]:
# ---- commit and push this run ----
import os, subprocess, sys
if "google.colab" in sys.modules or os.path.exists("/content"):
    MESSAGE = "Notebook 4 Baselines run"     # <-- edit if you like

    def sh(c):
        r = subprocess.run(c, shell=True, text=True, capture_output=True)
        print((r.stdout + r.stderr).strip()[:3000]); return r

    sh("git status --short")
    sh("git add -A")
    sh(f'git commit -m "{MESSAGE}"')
    r = sh("git push")
    if r.returncode != 0:
        print("\nPush failed. Usual causes: no GH_TOKEN secret, or the token "
              "lacks Contents: Read and write. Fix it and re-run this cell — "
              "the commit is already made locally, so nothing is lost until "
              "the runtime disconnects.")
else:
    print("Local run — commit with git as usual.")
